# typeshed_client smoke test

`typeshed_client` locates and parses the typeshed stub files (`.pyi`) it bundles for the standard library, and resolves names to where they are defined.

## Example 1 — the version, and the bundled stub for `os` is found

In [ ]:
import importlib.metadata
import typeshed_client

version = importlib.metadata.version("typeshed_client")
path = typeshed_client.get_stub_file("os")
assert path is not None and path.exists() and path.suffix == ".pyi", path
print("typeshed_client", version, "os stub:", path)

## Example 2 — the names a module's stub defines

In [ ]:
import typeshed_client

names = typeshed_client.get_stub_names("collections")
assert {"OrderedDict", "defaultdict", "deque"} <= set(names), sorted(names)[:20]
info = names["OrderedDict"]
assert info.is_exported
print(len(names), "names in collections; OrderedDict ->", type(info.ast).__name__)

## Example 3 — resolving a dotted name to where it is really defined

In [ ]:
import typeshed_client

resolver = typeshed_client.Resolver()
info = resolver.get_fully_qualified_name("os.path.join")
assert info is not None, "os.path.join not resolved"
# os.path is an alias of posixpath on Linux, so the answer is an ImportedInfo naming the real module.
print("os.path.join ->", type(info).__name__, getattr(info, "source_module", None))

## Example 4 — a function's parameters, read from its stub

In [ ]:
import ast
import typeshed_client

names = typeshed_client.get_stub_names("math")
node = names["gcd"].ast
funcs = [node] if isinstance(node, ast.FunctionDef) else [d.ast for d in node.definitions]
args = funcs[0].args
params = [a.arg for a in args.posonlyargs + args.args] + ([args.vararg.arg] if args.vararg else [])
assert params, ast.dump(funcs[0])
print("math.gcd parameters from the stub:", params)

## Example 5 — an unknown module gives None, not a crash

In [ ]:
import typeshed_client

assert typeshed_client.get_stub_file("no_such_module_xyz") is None
assert typeshed_client.get_stub_names("no_such_module_xyz") is None
print("unknown module: None, as documented")